In [5]:
from pathlib import Path
from zipfile import ZipFile

ruta_zip = Path("../data/raw/household_sensors.zip")

with ZipFile(ruta_zip, "r") as archivo_zip:
    nombres = archivo_zip.namelist()

print("Cantidad de elementos:", len(nombres))

for nombre in nombres[:20]:
    print(nombre)

Cantidad de elementos: 1595
sensordata/
sensordata/home100_livingroom1038_sensor4472_electric-mains_electric-combined.csv.gz
sensordata/home101_hall1045_sensor4512c4517_electric-mains_electric-combined.csv.gz
sensordata/home102_hall1053_sensor4571c4575_electric-mains_electric-combined.csv.gz
sensordata/home105_hall1111_sensor5093c5097_electric-mains_electric-combined.csv.gz
sensordata/home106_hall1086_sensor4891c4895_electric-mains_electric-combined.csv.gz
sensordata/home107_hall1076_sensor4745c4749_electric-mains_electric-combined.csv.gz
sensordata/home109_kitchen1091_sensor4999c5015_electric-mains_electric-combined.csv.gz
sensordata/home110_hall1094_sensor4956c4960_electric-mains_electric-combined.csv.gz
sensordata/home113_hall1201_sensor5922c5924_electric-mains_electric-combined.csv.gz
sensordata/home114_hall1141_sensor5356c5360_electric-mains_electric-combined.csv.gz
sensordata/home115_livingroom1146_sensor5464c5468_electric-mains_electric-combined.csv.gz
sensordata/home116_kitchen

In [8]:
import re
import pandas as pd

archivos_datos = [
    nombre for nombre in nombres
    if nombre.endswith(".csv.gz")
]

registros = []

for archivo in archivos_datos:
    coincidencia = re.search(r"home(\d+)", archivo)

    if coincidencia:
        registros.append({
            "archivo": archivo,
            "home_id": int(coincidencia.group(1)),
            "es_consumo_general": "electric-mains_electric-combined" in archivo
        })

catalogo = pd.DataFrame(registros)

print("Cantidad de archivos de datos:", len(catalogo))
print("Cantidad de viviendas:", catalogo["home_id"].nunique())
print(
    "Archivos de consumo eléctrico general:",
    catalogo["es_consumo_general"].sum()
)

catalogo.head()

Cantidad de archivos de datos: 1592
Cantidad de viviendas: 255
Archivos de consumo eléctrico general: 254


,archivo,home_id,es_consumo_general
0,sensordata/home100_livingroom1038_sensor4472_e...,100,True
1,sensordata/home101_hall1045_sensor4512c4517_el...,101,True
2,sensordata/home102_hall1053_sensor4571c4575_el...,102,True
3,sensordata/home105_hall1111_sensor5093c5097_el...,105,True
4,sensordata/home106_hall1086_sensor4891c4895_el...,106,True


In [9]:
# Filtramos únicamente los archivos que contienen
# el consumo eléctrico general de cada vivienda
consumo_general = catalogo[
    catalogo["es_consumo_general"] == True
].copy()

# Obtenemos la lista de viviendas disponibles,
# eliminamos repetidas y las ordenamos
viviendas_disponibles = (
    consumo_general["home_id"]
    .drop_duplicates()
    .sort_values()
)

# Seleccionamos 127 viviendas al azar
# random_state=42 hace que siempre se elijan las mismas
viviendas_seleccionadas = viviendas_disponibles.sample(
    n=127,
    random_state=42
).sort_values()

# Buscamos los archivos correspondientes
# a las viviendas seleccionadas
archivos_seleccionados = consumo_general[
    consumo_general["home_id"].isin(viviendas_seleccionadas)
].copy()

# Mostramos un resumen
print("Viviendas disponibles:", len(viviendas_disponibles))
print("Viviendas seleccionadas:", len(viviendas_seleccionadas))
print("Archivos seleccionados:", len(archivos_seleccionados))

# Mostramos las primeras filas
archivos_seleccionados.head()

Viviendas disponibles: 254
Viviendas seleccionadas: 127
Archivos seleccionados: 127


,archivo,home_id,es_consumo_general
2,sensordata/home102_hall1053_sensor4571c4575_el...,102,True
3,sensordata/home105_hall1111_sensor5093c5097_el...,105,True
6,sensordata/home109_kitchen1091_sensor4999c5015...,109,True
7,sensordata/home110_hall1094_sensor4956c4960_el...,110,True
12,sensordata/home117_hall1198_sensor5867c5871_el...,117,True


In [7]:
%pip install pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   -------------------------- ------------- 6.6/9.8 MB 36.6 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 33.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   -------------------------- ------------- 8.1/12.4 MB 42.1 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 32.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import gzip

# Tomamos el primer archivo de las 127 viviendas seleccionadas
primer_archivo = archivos_seleccionados.iloc[0]["archivo"]

print("Archivo que vamos a revisar:")
print(primer_archivo)

# Abrimos el archivo directamente desde el ZIP
# y leemos solamente las primeras 5 filas
with ZipFile(ruta_zip, "r") as archivo_zip:
    with archivo_zip.open(primer_archivo) as archivo_comprimido:
        with gzip.open(archivo_comprimido, mode="rt") as archivo_csv:
            muestra = pd.read_csv(archivo_csv, nrows=5)

muestra

Archivo que vamos a revisar:
sensordata/home102_hall1053_sensor4571c4575_electric-mains_electric-combined.csv.gz


,2017-03-09 10:25:53,0
0,2017-03-09 10:25:54,1067
1,2017-03-09 10:25:55,1036
2,2017-03-09 10:25:56,1024
3,2017-03-09 10:25:57,1016
4,2017-03-09 10:25:58,1012


In [11]:
# Volvemos a abrir el mismo archivo,
# indicando que no tiene nombres de columnas

with ZipFile(ruta_zip, "r") as archivo_zip:
    with archivo_zip.open(primer_archivo) as archivo_comprimido:
        with gzip.open(archivo_comprimido, mode="rt") as archivo_csv:
            muestra = pd.read_csv(
                archivo_csv,
                header=None,
                names=["fecha_hora", "potencia"]
            )

# Mostramos las primeras cinco filas
muestra.head()

,fecha_hora,potencia
0,2017-03-09 10:25:53,0
1,2017-03-09 10:25:54,1067
2,2017-03-09 10:25:55,1036
3,2017-03-09 10:25:56,1024
4,2017-03-09 10:25:57,1016


In [12]:
muestra.info()

<class 'pandas.DataFrame'>
RangeIndex: 34324507 entries, 0 to 34324506
Data columns (total 2 columns):
 #   Column      Dtype
---  ------      -----
 0   fecha_hora  str  
 1   potencia    int64
dtypes: int64(1), str(1)
memory usage: 523.8 MB


In [13]:
muestra.head(10)

,fecha_hora,potencia
0,2017-03-09 10:25:53,0
1,2017-03-09 10:25:54,1067
2,2017-03-09 10:25:55,1036
3,2017-03-09 10:25:56,1024
4,2017-03-09 10:25:57,1016
5,2017-03-09 10:25:58,1012
6,2017-03-09 10:25:59,1046
7,2017-03-09 10:26:00,1002
8,2017-03-09 10:26:01,1024
9,2017-03-09 10:26:02,1024


In [14]:
# Convertimos la fecha y hora de texto a formato datetime
muestra["fecha_hora"] = pd.to_datetime(
    muestra["fecha_hora"],
    errors="coerce"
)

# Revisamos si quedaron fechas inválidas
print("Fechas inválidas:", muestra["fecha_hora"].isna().sum())

muestra.info()

Fechas inválidas: 0
<class 'pandas.DataFrame'>
RangeIndex: 34324507 entries, 0 to 34324506
Data columns (total 2 columns):
 #   Column      Dtype         
---  ------      -----         
 0   fecha_hora  datetime64[us]
 1   potencia    int64         
dtypes: datetime64[us](1), int64(1)
memory usage: 523.8 MB


In [15]:
# Creamos una columna con la hora completa
muestra["hora"] = muestra["fecha_hora"].dt.floor("h")

# Calculamos la potencia promedio de cada hora
consumo_horario = (
    muestra.groupby("hora", as_index=False)
    .agg(potencia_promedio=("potencia", "mean"))
)

print("Filas originales:", len(muestra))
print("Filas después de resumir por hora:", len(consumo_horario))

consumo_horario.head()

Filas originales: 34324507
Filas después de resumir por hora: 10198


,hora,potencia_promedio
0,2017-03-09 10:00:00,588.806732
1,2017-03-09 11:00:00,517.519393
2,2017-03-09 12:00:00,1244.050961
3,2017-03-09 13:00:00,377.562848
4,2017-03-09 14:00:00,516.664089


In [16]:
print("Valores nulos:")
print(muestra.isna().sum())

print("\nDuplicados:")
print(muestra.duplicated().sum())

print("\nPotencias negativas:")
print((muestra["potencia"] < 0).sum())

print("\nResumen de potencia:")
print(muestra["potencia"].describe())

Valores nulos:
fecha_hora    0
potencia      0
hora          0
dtype: int64

Duplicados:
0

Potencias negativas:
0

Resumen de potencia:
count    3.432451e+07
mean     7.437857e+02
std      9.603898e+02
min      0.000000e+00
25%      1.990000e+02
50%      3.640000e+02
75%      6.090000e+02
max      1.863000e+04
Name: potencia, dtype: float64


In [17]:
del muestra

In [18]:
consumo_horario.to_csv(
    "../data/processed/prueba_consumo_horario.csv",
    index=False
)

In [19]:
# Identificamos la vivienda del archivo de prueba
home_id_prueba = archivos_seleccionados.iloc[0]["home_id"]

consumo_horario["home_id"] = home_id_prueba

In [20]:
# Creamos la fecha sin la hora
consumo_horario["fecha"] = consumo_horario["hora"].dt.date

# Extraemos la hora del día
consumo_horario["hora_dia"] = consumo_horario["hora"].dt.hour

# Indicamos si corresponde al horario pico
# Por ahora usamos de 18 a 22; después el equipo puede modificarlo
consumo_horario["es_horario_pico"] = consumo_horario[
    "hora_dia"
].between(18, 22)

# Resumimos por vivienda y día
consumo_diario = (
    consumo_horario
    .groupby(["home_id", "fecha"], as_index=False)
    .agg(
        potencia_promedio=("potencia_promedio", "mean"),
        potencia_maxima=("potencia_promedio", "max"),
        horas_registradas=("hora", "count")
    )
)

consumo_diario.head()

,home_id,fecha,potencia_promedio,potencia_maxima,horas_registradas
0,102,2017-03-09,771.788731,2908.171810,14
1,102,2017-03-10,523.829512,1296.598495,24
2,102,2017-03-11,451.380681,1532.653215,24
3,102,2017-03-12,909.852201,2830.515091,24
4,102,2017-03-13,636.947767,1429.069761,24


In [21]:
import gc

# Primero probamos solamente con 3 viviendas
archivos_prueba = archivos_seleccionados.head(3)

resultados_horarios = []

with ZipFile(ruta_zip, "r") as archivo_zip:

    for _, fila in archivos_prueba.iterrows():

        nombre_archivo = fila["archivo"]
        home_id = fila["home_id"]

        print(f"Procesando vivienda {home_id}...")

        bloques_horarios = []

        # Abrimos un archivo de vivienda dentro del ZIP
        with archivo_zip.open(nombre_archivo) as archivo_comprimido:
            with gzip.open(archivo_comprimido, mode="rt") as archivo_csv:

                # Leemos un millón de filas por vez
                for bloque in pd.read_csv(
                    archivo_csv,
                    header=None,
                    names=["fecha_hora", "potencia"],
                    chunksize=1_000_000
                ):

                    # Convertimos fecha y hora
                    bloque["fecha_hora"] = pd.to_datetime(
                        bloque["fecha_hora"],
                        errors="coerce"
                    )

                    # Eliminamos registros incompletos
                    bloque = bloque.dropna(
                        subset=["fecha_hora", "potencia"]
                    )

                    # Eliminamos potencias negativas
                    bloque = bloque[
                        bloque["potencia"] >= 0
                    ]

                    # Creamos la hora correspondiente
                    bloque["hora"] = (
                        bloque["fecha_hora"]
                        .dt.floor("h")
                    )

                    # Resumimos este bloque por hora
                    resumen_bloque = (
                        bloque.groupby("hora", as_index=False)
                        .agg(
                            suma_potencia=("potencia", "sum"),
                            cantidad_mediciones=("potencia", "count"),
                            potencia_maxima=("potencia", "max")
                        )
                    )

                    bloques_horarios.append(resumen_bloque)

        # Unimos los resultados parciales de la vivienda
        horario_vivienda = pd.concat(
            bloques_horarios,
            ignore_index=True
        )

        # Algunas horas aparecen en más de un bloque,
        # por eso las agrupamos nuevamente
        horario_vivienda = (
            horario_vivienda.groupby("hora", as_index=False)
            .agg(
                suma_potencia=("suma_potencia", "sum"),
                cantidad_mediciones=("cantidad_mediciones", "sum"),
                potencia_maxima=("potencia_maxima", "max")
            )
        )

        # Calculamos la potencia promedio de cada hora
        horario_vivienda["potencia_promedio"] = (
            horario_vivienda["suma_potencia"]
            / horario_vivienda["cantidad_mediciones"]
        )

        horario_vivienda["home_id"] = home_id

        resultados_horarios.append(
            horario_vivienda[
                [
                    "home_id",
                    "hora",
                    "potencia_promedio",
                    "potencia_maxima",
                    "cantidad_mediciones"
                ]
            ]
        )

        # Liberamos memoria antes de continuar
        del bloques_horarios
        gc.collect()

print("Prueba terminada")

Procesando vivienda 102...
Procesando vivienda 105...
Procesando vivienda 109...
Prueba terminada


In [22]:
consumo_horario_prueba = pd.concat(
    resultados_horarios,
    ignore_index=True
)

print("Cantidad de filas horarias:", len(consumo_horario_prueba))

consumo_horario_prueba.head()

Cantidad de filas horarias: 28118


,home_id,hora,potencia_promedio,potencia_maxima,cantidad_mediciones
0,102,2017-03-09 10:00:00,588.806732,2241,1842
1,102,2017-03-09 11:00:00,517.519393,1793,3429
2,102,2017-03-09 12:00:00,1244.050961,2875,3591
3,102,2017-03-09 13:00:00,377.562848,2028,3596
4,102,2017-03-09 14:00:00,516.664089,2768,3492


In [23]:
consumo_horario_prueba["fecha"] = (
    consumo_horario_prueba["hora"].dt.date
)

consumo_diario_prueba = (
    consumo_horario_prueba
    .groupby(["home_id", "fecha"], as_index=False)
    .agg(
        potencia_promedio=("potencia_promedio", "mean"),
        potencia_maxima=("potencia_maxima", "max"),
        horas_registradas=("hora", "count"),
        mediciones_totales=("cantidad_mediciones", "sum")
    )
)

print("Cantidad de filas diarias:", len(consumo_diario_prueba))

consumo_diario_prueba.head(10)

Cantidad de filas diarias: 1218


,home_id,fecha,potencia_promedio,potencia_maxima,horas_registradas,mediciones_totales
0,102,2017-03-09,771.788731,6428,14,45947
1,102,2017-03-10,523.829512,3374,24,81287
2,102,2017-03-11,451.380681,4245,24,84495
3,102,2017-03-12,909.852201,7431,24,84913
4,102,2017-03-13,636.947767,5645,24,80188
5,102,2017-03-14,416.226218,4609,24,79136
6,102,2017-03-15,650.120639,6372,24,80763
7,102,2017-03-16,463.516868,4835,24,86160
8,102,2017-03-17,633.092663,5684,24,83509
9,102,2017-03-18,710.290070,7827,24,79543


In [24]:
# Marcamos como completo un día que tenga las 24 horas registradas
consumo_diario_prueba["dia_completo"] = (
    consumo_diario_prueba["horas_registradas"] == 24
)

print(
    consumo_diario_prueba["dia_completo"]
    .value_counts()
)

consumo_diario_prueba.head()

dia_completo
True     1000
False     218
Name: count, dtype: int64


,home_id,fecha,potencia_promedio,potencia_maxima,horas_registradas,mediciones_totales,dia_completo
0,102,2017-03-09,771.788731,6428,14,45947,False
1,102,2017-03-10,523.829512,3374,24,81287,True
2,102,2017-03-11,451.380681,4245,24,84495,True
3,102,2017-03-12,909.852201,7431,24,84913,True
4,102,2017-03-13,636.947767,5645,24,80188,True


In [25]:
consumo_diario_prueba.to_csv(
    "../data/processed/prueba_3_viviendas_diario.csv",
    index=False
)

In [26]:
import gc
from pathlib import Path

# Carpeta donde se guardará el resultado diario de cada vivienda
carpeta_salida = Path("../data/processed/viviendas_diarias")
carpeta_salida.mkdir(parents=True, exist_ok=True)

with ZipFile(ruta_zip, "r") as archivo_zip:

    for numero, (_, fila) in enumerate(
        archivos_seleccionados.iterrows(),
        start=1
    ):

        nombre_archivo = fila["archivo"]
        home_id = fila["home_id"]

        ruta_salida = carpeta_salida / f"home_{home_id}_diario.csv"

        # Si esa vivienda ya fue procesada, la salta
        if ruta_salida.exists():
            print(f"[{numero}/127] Vivienda {home_id} ya procesada")
            continue

        print(f"[{numero}/127] Procesando vivienda {home_id}...")

        bloques_horarios = []

        with archivo_zip.open(nombre_archivo) as archivo_comprimido:
            with gzip.open(archivo_comprimido, mode="rt") as archivo_csv:

                # Lee el archivo de a un millón de filas
                for bloque in pd.read_csv(
                    archivo_csv,
                    header=None,
                    names=["fecha_hora", "potencia"],
                    chunksize=1_000_000
                ):

                    # Convierte fecha y potencia
                    bloque["fecha_hora"] = pd.to_datetime(
                        bloque["fecha_hora"],
                        errors="coerce"
                    )

                    bloque["potencia"] = pd.to_numeric(
                        bloque["potencia"],
                        errors="coerce"
                    )

                    # Elimina filas incompletas
                    bloque = bloque.dropna(
                        subset=["fecha_hora", "potencia"]
                    )

                    # Elimina potencias negativas
                    bloque = bloque[
                        bloque["potencia"] >= 0
                    ]

                    # Agrupa las mediciones por hora
                    bloque["hora"] = (
                        bloque["fecha_hora"]
                        .dt.floor("h")
                    )

                    resumen_bloque = (
                        bloque.groupby("hora", as_index=False)
                        .agg(
                            suma_potencia=("potencia", "sum"),
                            cantidad_mediciones=("potencia", "count"),
                            potencia_maxima=("potencia", "max")
                        )
                    )

                    bloques_horarios.append(resumen_bloque)

        # Une todos los bloques de una misma vivienda
        horario_vivienda = pd.concat(
            bloques_horarios,
            ignore_index=True
        )

        # Algunas horas pueden aparecer en más de un bloque
        horario_vivienda = (
            horario_vivienda.groupby("hora", as_index=False)
            .agg(
                suma_potencia=("suma_potencia", "sum"),
                cantidad_mediciones=("cantidad_mediciones", "sum"),
                potencia_maxima=("potencia_maxima", "max")
            )
        )

        # Calcula la potencia promedio por hora
        horario_vivienda["potencia_promedio"] = (
            horario_vivienda["suma_potencia"]
            / horario_vivienda["cantidad_mediciones"]
        )

        horario_vivienda["home_id"] = home_id
        horario_vivienda["fecha"] = (
            horario_vivienda["hora"].dt.date
        )

        # Resume la vivienda por día
        diario_vivienda = (
            horario_vivienda
            .groupby(["home_id", "fecha"], as_index=False)
            .agg(
                potencia_promedio=("potencia_promedio", "mean"),
                potencia_maxima=("potencia_maxima", "max"),
                horas_registradas=("hora", "count"),
                mediciones_totales=("cantidad_mediciones", "sum")
            )
        )

        # Marca si el día tiene las 24 horas
        diario_vivienda["dia_completo"] = (
            diario_vivienda["horas_registradas"] == 24
        )

        # Guarda un archivo por vivienda
        diario_vivienda.to_csv(
            ruta_salida,
            index=False
        )

        # Libera memoria antes de seguir
        del bloques_horarios
        del horario_vivienda
        del diario_vivienda
        gc.collect()

print("Procesamiento terminado")

[1/127] Procesando vivienda 102...
[2/127] Procesando vivienda 105...
[3/127] Procesando vivienda 109...
[4/127] Procesando vivienda 110...
[5/127] Procesando vivienda 117...
[6/127] Procesando vivienda 121...
[7/127] Procesando vivienda 122...
[8/127] Procesando vivienda 126...
[9/127] Procesando vivienda 135...
[10/127] Procesando vivienda 136...
[11/127] Procesando vivienda 137...
[12/127] Procesando vivienda 138...
[13/127] Procesando vivienda 139...
[14/127] Procesando vivienda 144...
[15/127] Procesando vivienda 146...
[16/127] Procesando vivienda 147...
[17/127] Procesando vivienda 148...
[18/127] Procesando vivienda 149...
[19/127] Procesando vivienda 150...
[20/127] Procesando vivienda 153...
[21/127] Procesando vivienda 155...
[22/127] Procesando vivienda 156...
[23/127] Procesando vivienda 157...
[24/127] Procesando vivienda 161...
[25/127] Procesando vivienda 164...
[26/127] Procesando vivienda 166...
[27/127] Procesando vivienda 167...
[28/127] Procesando vivienda 168...
[

In [27]:
from pathlib import Path

carpeta_salida = Path("../data/processed/viviendas_diarias")

archivos_generados = list(carpeta_salida.glob("home_*_diario.csv"))

print("Archivos generados:", len(archivos_generados))

Archivos generados: 127


In [28]:
dataframes = []

for archivo in archivos_generados:
    df_vivienda = pd.read_csv(archivo)
    dataframes.append(df_vivienda)

consumo_diario_127 = pd.concat(
    dataframes,
    ignore_index=True
)

print("Filas totales:", len(consumo_diario_127))
print("Viviendas únicas:", consumo_diario_127["home_id"].nunique())

consumo_diario_127.head()

Filas totales: 33351
Viviendas únicas: 127


,home_id,fecha,potencia_promedio,potencia_maxima,horas_registradas,mediciones_totales,dia_completo
0,102,2017-03-09,771.788731,6428,14,45947,False
1,102,2017-03-10,523.829512,3374,24,81287,True
2,102,2017-03-11,451.380681,4245,24,84495,True
3,102,2017-03-12,909.852201,7431,24,84913,True
4,102,2017-03-13,636.947767,5645,24,80188,True


In [29]:
print("Duplicados:", consumo_diario_127.duplicated().sum())

print("\nValores nulos:")
print(consumo_diario_127.isna().sum())

print("\nDías completos e incompletos:")
print(consumo_diario_127["dia_completo"].value_counts())

Duplicados: 0

Valores nulos:
home_id               0
fecha                 0
potencia_promedio     0
potencia_maxima       0
horas_registradas     0
mediciones_totales    0
dia_completo          0
dtype: int64

Días completos e incompletos:
dia_completo
True     25495
False     7856
Name: count, dtype: int64


In [30]:
consumo_diario_127.to_csv(
    "../data/processed/ideal_127_viviendas_diario.csv",
    index=False
)

In [34]:
consumo_diario_127.to_parquet(
    "../data/processed/ideal_127_viviendas_diario.parquet",
    index=False
)

ArrowKeyError: A type extension with name pandas.period already defined

In [32]:
%pip install pyarrow

   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   -------- ------------------------------- 6.0/27.9 MB 37.0 MB/s eta 0:00:01
   --------------------- ------------------ 14.7/27.9 MB 38.4 MB/s eta 0:00:01
   ------------------------------ --------- 21.5/27.9 MB 37.8 MB/s eta 0:00:01
   ---------------------------------------  27.8/27.9 MB 37.5 MB/s eta 0:00:01
   ---------------------------------------  27.8/27.9 MB 37.5 MB/s eta 0:00:01
   ---------------------------------------- 27.9/27.9 MB 26.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
consumo_diario_127.to_parquet(
    "../data/processed/ideal_127_viviendas_diario.parquet",
    index=False
)

ArrowKeyError: A type extension with name pandas.period already defined

In [1]:
import pandas as pd

consumo_diario_127 = pd.read_csv(
    "../data/processed/ideal_127_viviendas_diario.csv"
)

print(consumo_diario_127.shape)
consumo_diario_127.head()

(33351, 7)


,home_id,fecha,potencia_promedio,potencia_maxima,horas_registradas,mediciones_totales,dia_completo
0,102,2017-03-09,771.788731,6428,14,45947,False
1,102,2017-03-10,523.829512,3374,24,81287,True
2,102,2017-03-11,451.380681,4245,24,84495,True
3,102,2017-03-12,909.852201,7431,24,84913,True
4,102,2017-03-13,636.947767,5645,24,80188,True


In [2]:
consumo_diario_127.to_parquet(
    "../data/processed/ideal_127_viviendas_diario.parquet",
    index=False,
    engine="pyarrow"
)

print("Archivo Parquet guardado correctamente")

Archivo Parquet guardado correctamente
